# Notebook 3 - Convolutional Neural Networks

**UKACM Autumn School: AI for Computational Mechanics**

### Where Notebook 2 left off

Notebook 2 finished with a test. A fully connected network was trained on the raw 64x64
microstructures, then retrained on the same images with one fixed random permutation applied to
every pixel position. The two scores were the same. The network could not tell a microstructure from
a bag of pixels, because a dense layer has no notion of which pixels are neighbours.

This notebook builds the layer that does have that notion, and then checks whether it helps.

### What you will do

1. **Part 1.** Convolution from first principles on toy images. Kernels, padding, stride, pooling,
   weight sharing. Nothing is trained. Write your own kernel and watch what it detects.
2. **Part 2.** Train a small CNN on the microstructures to predict $E_{mean}$, then run the
   permutation test from Notebook 2 on it.
3. **Part 3.** Train the same CNN on the anisotropy $\Delta E$ and compare it against the
   descriptor baseline of Notebook 2, which reads 14 named descriptors computed from the same
   64x64 images. The result is not the one you might expect.
4. **Part 4.** Look at what the trained network learned: first layer filters, feature maps, and a
   saliency map.

### Table of contents

| Part | Question |
|---|---|
| 1 | What does a convolution actually compute? |
| 2 | Does a CNN notice when the pixels are shuffled? |
| 3 | Does the CNN beat 14 hand-built descriptors on anisotropy? |
| 4 | What did the filters learn? |


In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first.

import os, io, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Text, Checkbox

import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)          # matches a free Colab CPU runtime

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})

C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("numpy", np.__version__, "| torch", torch.__version__, "| device", DEVICE)
if DEVICE == "cpu":
    print("No GPU found. Every timing quoted below is for 2 CPU cores.")


### Loading the data

Four files are needed, the same ones as Notebook 2:

- `microstructure_labels.csv`, one row per microstructure
- `microstructures_64.npz`, the binary images at 64x64
- `microstructure_descriptors_core14_64.npz`, the 14 named descriptors used in Notebook 2
- `microstructure_descriptors_v17_30_64.npz`, the full research bank of 30

Both descriptor files are computed from the same 64x64 images the CNN is given, so every comparison
in Part 3 is at one resolution and the inputs differ only in representation.

The next cell downloads and extracts the dataset automatically.

In [ ]:
# --- Load data ---------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/CEMS-Lab/autumn-school/main/datasets/machine_learning/NB3_data.zip"
DATA_DIR = "."

FILES = ["microstructure_labels.csv", "microstructures_64.npz",
         "microstructure_descriptors_core14_64.npz",
         "microstructure_descriptors_v17_30_64.npz"]

def _have():
    return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES)

if not _have() and DATA_URL:
    print("Downloading ...")
    with urllib.request.urlopen(DATA_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)

if not _have():
    try:
        from google.colab import files
        print("Select:", ", ".join(FILES))
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            "Put " + ", ".join(FILES) + " beside the notebook, or set DATA_URL above.")

df = pd.read_csv(os.path.join(DATA_DIR, "microstructure_labels.csv"))
df["E_mean"] = (df.E22 + df.E33) / 2
df["dE"]     =  df.E22 - df.E33
df["vf"]     =  df.vol_frac / 100.0

clean = df[~df.outlier_flag].copy().reset_index(drop=True)

_img = np.load(os.path.join(DATA_DIR, "microstructures_64.npz"), allow_pickle=True)
IMAGES, IMG_KEYS = _img["images"], list(_img["keys"])
IMG_IDX = {k: i for i, k in enumerate(IMG_KEYS)}

_dsc = np.load(os.path.join(DATA_DIR, "microstructure_descriptors_core14_64.npz"),
               allow_pickle=True)
DESC_ALL, DESC_KEYS = _dsc["D"], list(_dsc["keys"])
DESC_NAMES    = [str(v) for v in _dsc["names"]]
DESC_MEANINGS = [str(v) for v in _dsc["meanings"]]
DESC_IDX = {k: i for i, k in enumerate(DESC_KEYS)}

_d30 = np.load(os.path.join(DATA_DIR, "microstructure_descriptors_v17_30_64.npz"),
               allow_pickle=True)
D30_ALL, D30_KEYS = _d30["D"], list(_d30["keys"])
D30_NAMES = [str(v) for v in _d30["names"]]
D30_IDX = {k: i for i, k in enumerate(D30_KEYS)}

X_IMG  = IMAGES[[IMG_IDX[k]  for k in clean.key]].astype(np.float32)
X_DESC = DESC_ALL[[DESC_IDX[k] for k in clean.key]].astype(np.float32)
X_D30  = D30_ALL[[D30_IDX[k]  for k in clean.key]].astype(np.float32)
y_mean = clean["E_mean"].values.astype(np.float32)
y_dE   = clean["dE"].values.astype(np.float32)

print(f"{len(clean)} microstructures kept, {int(df.outlier_flag.sum())} outliers dropped")
print(f"images      {X_IMG.shape}   values in {{{X_IMG.min():.0f}, {X_IMG.max():.0f}}}")
print(f"descriptors {X_DESC.shape} (named)   {X_D30.shape} (full bank)")
print()
print("the 14 named descriptors Notebook 2 teaches")
for _n, _m in zip(DESC_NAMES, DESC_MEANINGS):
    print(f"   {_n:20s} {_m}")


---

# Part 1 - Convolution from first principles

Nothing in this part is trained. The point is to know exactly what the layer computes before asking
a network to learn one.

A convolution takes a small array of weights, the **kernel**, slides it over the input, and at each
position writes the sum of the elementwise products into an output array called the **feature map**:

$$\boxed{\;O(i,j) \;=\; \sum_{m=0}^{k-1}\ \sum_{n=0}^{k-1} I(i+m,\; j+n)\; K(m,n)\;}$$

The symbols:

| Symbol | Shape | Meaning |
|---|---|---|
| $I$ | $H \times W$ | the input image, here 0 for matrix and 1 for fibre |
| $K$ | $k \times k$ | the kernel, $k = 3$ throughout Part 1 |
| $O$ | $H_{out} \times W_{out}$ | the feature map, one number per window position |
| $i, j$ | | the position of the window's top-left corner in the input |
| $m, n$ | $0 \ldots k-1$ | the position inside the window |

One output value is $k^2$ multiplications and $k^2 - 1$ additions. Nothing else happens.

Two properties follow immediately from that formula, and they are the whole reason the layer exists.

- **Locality.** Each output depends only on a $k \times k$ neighbourhood of the input. Neighbouring
  pixels are treated as neighbours by construction.
- **Weight sharing.** The same nine numbers are used at every position. A pattern detected in one
  corner is detected the same way in the other corner, and the parameter count does not grow with
  the image.

### Convolution or cross-correlation

Strictly, the formula above is a **cross-correlation**. A true convolution flips the kernel in both
axes before the sum,

$$(I * K)(i,j) \;=\; \sum_{m}\sum_{n} I(i-m,\; j-n)\; K(m,n)$$

which is the same operation applied to $K$ rotated by 180 degrees. PyTorch, TensorFlow and every
other deep learning library implement the cross-correlation and call it convolution. Since the nine
numbers in $K$ are learned rather than prescribed, the network simply learns the flipped kernel where
it needs one, and the distinction has no consequence. It matters only when you are comparing against
a classical signal-processing result that assumes the flip.

### A test image that looks like the data

Everything in Part 1 runs on a synthetic two-phase image: discs of fibre on a matrix background,
periodic across the boundaries, in the same style as the real microstructures.

In [ ]:
# --- A synthetic microstructure, and a small crop of it ----------------------
def synth_microstructure(n=64, discs=((16, 16, 7), (17, 44, 9), (46, 22, 8),
                                      (44, 50, 6), (32, 32, 5), (58, 60, 7))):
    yy, xx = np.mgrid[0:n, 0:n]
    img = np.zeros((n, n), dtype=np.float32)
    for cy, cx, r in discs:
        dy = np.minimum(np.abs(yy - cy), n - np.abs(yy - cy))   # periodic distance
        dx = np.minimum(np.abs(xx - cx), n - np.abs(xx - cx))
        img[dy**2 + dx**2 <= r*r] = 1.0
    return img

SYN = synth_microstructure()

# a 12x12 block-averaged crop, used for the step-by-step arithmetic and the animation
SMALL = SYN[8:32, 8:32].reshape(12, 2, 12, 2).mean(axis=(1, 3)).round(1).astype(np.float32)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(SYN, cmap="gray", interpolation="nearest")
axes[0].set_title(f"synthetic test image 64x64\nfibre fraction {SYN.mean():.3f}", fontsize=9)
axes[1].imshow(X_IMG[0], cmap="gray", interpolation="nearest")
axes[1].set_title(f"a real microstructure\nfibre fraction {X_IMG[0].mean():.3f}", fontsize=9)
axes[2].imshow(SMALL, cmap="gray", interpolation="nearest")
axes[2].set_title("the 12x12 crop used below", fontsize=9)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()


### 1a. The arithmetic, position by position

Take a 3x3 vertical edge kernel

$$K = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1\end{bmatrix}$$

and place it at three positions on the crop. The cell prints every product and their sum, so there
is nothing hidden.

A kernel like this returns near zero inside a uniform region, because the positive and negative
weights cancel. It returns a large value where the intensity changes from left to right, which on a
two-phase image means a fibre boundary.

In [ ]:
# --- Convolution arithmetic written out in full ------------------------------
K_EDGE_V = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=np.float32)

def conv2d_valid(img, k):
    kh, kw = k.shape
    oh, ow = img.shape[0] - kh + 1, img.shape[1] - kw + 1
    out = np.empty((oh, ow), dtype=np.float32)
    for i in range(oh):
        for j in range(ow):
            out[i, j] = float((img[i:i+kh, j:j+kw] * k).sum())
    return out

for (i, j) in [(0, 0), (4, 3), (5, 8)]:
    patch = SMALL[i:i+3, j:j+3]
    prod  = patch * K_EDGE_V
    terms = " + ".join(f"({p:g}x{w:+g})" for p, w in zip(patch.ravel(), K_EDGE_V.ravel()))
    print(f"position (i={i}, j={j})")
    print(f"  patch  = {patch.ravel()}")
    print(f"  O = {terms}")
    print(f"  O({i},{j}) = {prod.sum():+.1f}\n")

FMAP_EDGE = conv2d_valid(SMALL, K_EDGE_V)
print(f"input {SMALL.shape} with a 3x3 kernel and no padding gives {FMAP_EDGE.shape}")


In [ ]:
# --- Schematic: one output element, with its arithmetic ----------------------
I0, J0 = 4, 0                                   # the window position drawn below
patch0 = SMALL[I0:I0+3, J0:J0+3]
FM0    = conv2d_valid(SMALL, K_EDGE_V)

fig, (axA, axB, axC) = plt.subplots(1, 3, figsize=(13, 4.4),
                                    gridspec_kw={"width_ratios": [1.0, 0.95, 1.0]})

# --- panel A: the input, with the window marked ------------------------------
axA.imshow(SMALL, cmap="gray", interpolation="nearest", vmin=0, vmax=1)
for g in range(SMALL.shape[0] + 1):
    axA.axhline(g - 0.5, color="0.55", lw=0.4)
    axA.axvline(g - 0.5, color="0.55", lw=0.4)
axA.add_patch(Rectangle((J0 - 0.5, I0 - 0.5), 3, 3, fill=False, ec=C_FIT, lw=3.0))
axA.annotate("$(i, j)$", xy=(J0 - 0.4, I0 - 0.4), xytext=(J0 + 2.6, I0 - 3.2),
             fontsize=11, color=C_FIT, ha="center",
             bbox=dict(fc="w", ec="none", alpha=0.85, boxstyle="round,pad=0.2"),
             arrowprops=dict(arrowstyle="->", lw=1.4, color=C_FIT))
axA.set_title("input $I$, window at $(i,j) = (" + str(I0) + ", " + str(J0) + ")$",
              fontsize=10)
axA.set_xticks([]); axA.set_yticks([]); axA.grid(False)

# --- panel B: the patch and the kernel, entry by entry ------------------------
axB.set_xlim(-0.75, 3.05); axB.set_ylim(3.05, -0.75)
axB.grid(False); axB.set_xticks([]); axB.set_yticks([])
for a_ in ("top", "right", "bottom", "left"):
    axB.spines[a_].set_visible(False)
for r in range(3):
    for c in range(3):
        axB.add_patch(Rectangle((c, r), 1, 1, fc="0.965", ec="0.55", lw=1.0))
        axB.text(c + 0.5, r + 0.34, f"{patch0[r, c]:g}", ha="center", va="center",
                 fontsize=15, color="k")
        axB.text(c + 0.5, r + 0.72, f"$\\times$ {K_EDGE_V[r, c]:+g}", ha="center",
                 va="center", fontsize=11, color=C_FIT)
for c in range(3):
    axB.text(c + 0.5, -0.30, f"$n={c}$", ha="center", va="center", fontsize=10,
             color="0.35")
for r in range(3):
    axB.text(-0.36, r + 0.5, f"$m={r}$", ha="center", va="center", fontsize=10,
             color="0.35")
axB.set_title("$I(i{+}m,\\, j{+}n)$ in black\n$K(m,n)$ in orange", fontsize=10)

# --- panel C: the feature map, with the one cell this fills in ---------------
mm = np.abs(FM0).max()
axC.imshow(FM0, cmap="coolwarm", interpolation="nearest", vmin=-mm, vmax=mm)
for g in range(FM0.shape[0] + 1):
    axC.axhline(g - 0.5, color="0.55", lw=0.4)
    axC.axvline(g - 0.5, color="0.55", lw=0.4)
axC.add_patch(Rectangle((J0 - 0.5, I0 - 0.5), 1, 1, fill=False, ec=C_FIT, lw=3.0))
axC.annotate("$O(i,j)$", xy=(J0 + 0.4, I0 - 0.4), xytext=(J0 + 3.0, I0 - 2.8), fontsize=11,
             color=C_FIT, ha="center",
             bbox=dict(fc="w", ec="none", alpha=0.85, boxstyle="round,pad=0.2"),
             arrowprops=dict(arrowstyle="->", lw=1.4, color=C_FIT))
axC.set_title("feature map $O$", fontsize=10)
axC.set_xticks([]); axC.set_yticks([]); axC.grid(False)

terms0 = " + ".join(f"({p:g})({w:+g})" for p, w in zip(patch0.ravel(), K_EDGE_V.ravel()))
line1 = "$O(i,j) = \\sum_{m=0}^{2}\\sum_{n=0}^{2} I(i{+}m,\\, j{+}n)\\, K(m,n)$"
fig.text(0.5, -0.02, line1, ha="center", fontsize=12)
fig.text(0.5, -0.10, "= " + terms0 + " = " + f"{FM0[I0, J0]:+.1f}",
         ha="center", fontsize=11, color=C_FIT)
plt.tight_layout(); plt.show()


**What the schematic shows.** Left: the whole input with one $3 \times 3$ window outlined, its
top-left corner at $(i,j)$. Middle: that window blown up, with the input value $I(i+m, j+n)$ in black
and the kernel weight $K(m,n)$ in orange in each cell, indexed exactly as in the equation. Right: the
feature map, with the single cell that this one window fills in.

The line underneath is the sum written out. Nine products, one addition, one number. The window then
moves one column to the right and the same nine weights are used again, which is weight sharing.

### 1b. The kernel sweeping across the image

The animation below is the same arithmetic run over every position, on an 8x8 crop so that all 36
window positions get a frame and nothing is skipped.

Left: the input, with the current window outlined and its corner at $(i,j)$. Middle: the nine input
values and the nine kernel weights, indexed $m$ and $n$ as in the equation, with the resulting
$O(i,j)$ underneath. Right: the feature map filling in, grey where a cell has not been computed yet.

Watch where the output goes strongly positive and strongly negative. The vertical edge kernel fires
with one sign on the left edge of a fibre and the other sign on the right edge, and returns almost
nothing in the interior of either phase.

In [ ]:
# --- Animation: the kernel sweeping across the input -------------------------
# An 8x8 crop, so every one of the 36 window positions gets its own frame and
# nothing is skipped.
IN_A  = SMALL[1:9, 0:8]
K_A   = K_EDGE_V
OH, OW = IN_A.shape[0] - 2, IN_A.shape[1] - 2
POS    = [(i, j) for i in range(OH) for j in range(OW)]
N_FR   = len(POS)

FULL_A = conv2d_valid(IN_A, K_A)
vmax_a = np.abs(FULL_A).max()

cmap_fm = plt.cm.coolwarm.copy()
cmap_fm.set_bad("0.90")                      # cells not yet computed

fig, (axL, axM, axR) = plt.subplots(1, 3, figsize=(13, 4.6),
                                    gridspec_kw={"width_ratios": [1.0, 1.0, 1.0]})
for a in (axL, axM, axR):
    a.grid(False); a.set_xticks([]); a.set_yticks([])

# --- left: the input, with the current window -------------------------------
axL.imshow(IN_A, cmap="gray", interpolation="nearest", vmin=0, vmax=1)
for g in range(IN_A.shape[0] + 1):
    axL.axhline(g - 0.5, color="0.55", lw=0.5); axL.axvline(g - 0.5, color="0.55", lw=0.5)
box = Rectangle((-0.5, -0.5), 3, 3, fill=False, ec=C_FIT, lw=3.0, zorder=5)
axL.add_patch(box)
axL.set_title("input $I$", fontsize=10)

# --- middle: the patch, the kernel and the running arithmetic ---------------
axM.set_xlim(-0.75, 3.05); axM.set_ylim(3.6, -0.55)
for a_ in ("top", "right", "bottom", "left"):
    axM.spines[a_].set_visible(False)
txt_I, txt_K = [], []
for r in range(3):
    for c in range(3):
        axM.add_patch(Rectangle((c, r), 1, 1, fc="0.965", ec="0.55", lw=1.0))
        txt_I.append(axM.text(c + 0.5, r + 0.34, "", ha="center", va="center",
                              fontsize=15, color="k"))
        txt_K.append(axM.text(c + 0.5, r + 0.72, f"$\\times$ {K_A[r, c]:+g}",
                              ha="center", va="center", fontsize=11, color=C_FIT))
for c in range(3):
    axM.text(c + 0.5, -0.28, f"$n={c}$", ha="center", fontsize=9, color="0.35")
for r in range(3):
    axM.text(-0.36, r + 0.5, f"$m={r}$", ha="center", va="center", fontsize=9, color="0.35")
sum_txt = axM.text(1.5, 3.35, "", ha="center", va="center", fontsize=13, color=C_FIT)
axM.set_title("$I(i{+}m,\\,j{+}n)$ black,  $K(m,n)$ orange", fontsize=10)

# --- right: the feature map filling in --------------------------------------
fmap = np.full((OH, OW), np.nan, dtype=np.float32)
imR = axR.imshow(fmap, cmap=cmap_fm, interpolation="nearest", vmin=-vmax_a, vmax=vmax_a)
for g in range(OH + 1):
    axR.axhline(g - 0.5, color="0.55", lw=0.5); axR.axvline(g - 0.5, color="0.55", lw=0.5)
cur = Rectangle((-0.5, -0.5), 1, 1, fill=False, ec=C_FIT, lw=3.0, zorder=5)
axR.add_patch(cur)
axR.set_title("feature map $O$", fontsize=10)
plt.colorbar(imR, ax=axR, shrink=0.8, label="$O(i,j)$")

def update_conv(f):
    fmap[:] = np.nan
    for k in range(f + 1):
        i_, j_ = POS[k]
        fmap[i_, j_] = float((IN_A[i_:i_+3, j_:j_+3] * K_A).sum())
    i, j = POS[f]
    box.set_xy((j - 0.5, i - 0.5))
    cur.set_xy((j - 0.5, i - 0.5))
    imR.set_data(fmap)
    patch = IN_A[i:i+3, j:j+3]
    for t, v in zip(txt_I, patch.ravel()):
        t.set_text(f"{v:g}")
    sum_txt.set_text(f"$O({i},{j})$ = {fmap[i, j]:+.1f}")
    axL.set_title(f"input $I$, window at $(i,j) = ({i}, {j})$", fontsize=10)
    return [imR, box, cur, sum_txt] + txt_I

anim_conv = animation.FuncAnimation(fig, update_conv, frames=N_FR, interval=260, blit=False)
plt.close(fig)
HTML(anim_conv.to_jshtml())


**What the animation shows.** The kernel never changes. The same nine orange numbers are used
at every one of the 36 positions, and the only thing that varies is the nine black numbers under
them. That is weight sharing, seen directly.

The feature map ends up red on one side of each fibre and blue on the other, with near zero through
the middle of the fibre and through the matrix. A kernel whose weights sum to zero cancels on any
uniform patch, so only the boundaries survive, and the sign records which way the intensity was
changing.

### 1c. What different kernels detect

Fixed kernels of this kind predate machine learning by decades. The change a CNN brings is that the
nine numbers are not written down by a person, they are fitted by gradient descent along with
everything else.

The gallery below applies six fixed kernels to the synthetic microstructure. Each acts on the same
input and each answers a different question about it.

In [ ]:
# --- A gallery of fixed kernels ----------------------------------------------
KERNELS = {
    "identity":        np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], np.float32),
    "blur (3x3 mean)": np.ones((3, 3), np.float32) / 9.0,
    "vertical edge":   K_EDGE_V,
    "horizontal edge": K_EDGE_V.T,
    "Laplacian":       np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], np.float32),
    "diagonal":        np.array([[-2, -1, 0], [-1, 0, 1], [0, 1, 2]], np.float32),
}

fig, axes = plt.subplots(2, 4, figsize=(14, 6.4))
axes = axes.ravel()
axes[0].imshow(SYN, cmap="gray", interpolation="nearest")
axes[0].set_title("input", fontsize=9)
for ax, (name, k) in zip(axes[1:7], KERNELS.items()):
    out = conv2d_valid(SYN, k)
    m = np.abs(out).max() + 1e-9
    ax.imshow(out, cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    ax.set_title(f"{name}\nrange {out.min():+.1f} to {out.max():+.1f}", fontsize=9)
axes[7].axis("off")
for a in axes[:7]:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()


### 1d. Write your own kernel

Type nine numbers, separated by commas or spaces, and the figure redraws. The preset dropdown fills
the box for you, then edit it.

Things worth trying:

- A kernel whose entries sum to zero returns zero on any uniform region, so only boundaries survive.
- A kernel whose entries sum to one preserves the average intensity, which is what a blur does.
- Swap the sign of every entry and the feature map flips colour. A ReLU after the convolution would
  then keep the opposite set of boundaries, which is why networks learn filters in near-opposite
  pairs.

In [ ]:
# --- Interactive kernel editor ------------------------------------------------
PRESETS = {"custom": None,
           "vertical edge":   "-1,0,1, -2,0,2, -1,0,1",
           "horizontal edge": "-1,-2,-1, 0,0,0, 1,2,1",
           "Laplacian":       "0,-1,0, -1,4,-1, 0,-1,0",
           "blur":            "0.111,0.111,0.111, 0.111,0.111,0.111, 0.111,0.111,0.111",
           "sharpen":         "0,-1,0, -1,5,-1, 0,-1,0"}

def kernel_editor(preset="vertical edge", values="-1,0,1, -2,0,2, -1,0,1"):
    text = PRESETS[preset] if PRESETS[preset] is not None else values
    try:
        nums = [float(v) for v in text.replace(",", " ").split()]
        if len(nums) != 9:
            raise ValueError
        k = np.array(nums, dtype=np.float32).reshape(3, 3)
    except ValueError:
        print("Give exactly nine numbers, separated by commas or spaces.")
        return

    out = conv2d_valid(SYN, k)
    m = np.abs(out).max() + 1e-9

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 4),
                             gridspec_kw={"width_ratios": [1, 0.55, 1]})
    axes[0].imshow(SYN, cmap="gray", interpolation="nearest")
    axes[0].set_title("input", fontsize=10)

    axes[1].imshow(k, cmap="coolwarm", vmin=-np.abs(k).max() - 1e-9,
                   vmax=np.abs(k).max() + 1e-9, interpolation="nearest")
    for r in range(3):
        for c in range(3):
            axes[1].text(c, r, f"{k[r, c]:g}", ha="center", va="center", fontsize=12)
    axes[1].set_title(f"kernel, sum = {k.sum():+.3f}", fontsize=10)

    axes[2].imshow(out, cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    axes[2].set_title(f"feature map {out.shape}\nrange {out.min():+.2f} to {out.max():+.2f}",
                      fontsize=10)
    for a in axes:
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
    plt.tight_layout(); plt.show()

interact(kernel_editor,
         preset=Dropdown(options=list(PRESETS.keys()), value="vertical edge"),
         values=Text(value="-1,0,1, -2,0,2, -1,0,1", description="9 values",
                     continuous_update=False));


### 1e. Padding and stride, and the output shape

Without padding the feature map is smaller than the input, and the border pixels take part in fewer
products than the interior ones. Padding adds a border of extra values before the convolution. Stride
is how far the window jumps between positions.

For an input of side $n$, kernel size $k$, padding $p$ and stride $s$, the feature map has side

$$\boxed{\;n_{out} \;=\; \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1\;}$$

$n$ is the input side in pixels, $k$ the kernel side, $p$ the number of pixels added on **each** side
of the input, which is why it appears as $2p$, and $s$ the stride, the number of pixels the window
jumps between positions. The floor is there because a window that would hang over the edge is simply
not evaluated.

Three cases worth memorising: $k=3, p=1, s=1$ leaves the size unchanged, $p=0$ loses $k-1$ pixels,
and $s=2$ halves it.

### Translation equivariance, and why the padding must be circular

Convolution commutes with translation. Write $T_{\mathbf{u}}$ for a shift of the image by
$\mathbf{u} = (u_1, u_2)$ pixels. Then

$$\bigl(T_{\mathbf{u}} I\bigr) * K \;=\; T_{\mathbf{u}}\bigl(I * K\bigr)$$

Shift the input and the feature map shifts by the same amount, unchanged in every other respect. That
is **equivariance**, and it is the formal version of "the same pattern is detected wherever it
occurs". It is exact only in the interior. At the boundary the answer depends on what the padding
claims is outside the image.

The microstructures in this dataset are **periodic unit cells**. The finite element homogenisation
that produced the labels applied periodic boundary conditions, so a fibre leaving the right edge
genuinely re-enters on the left, and the same in $y$. Circular padding, `padding_mode="circular"`,
wraps the image round and reproduces exactly that. Zero padding instead asserts a border of empty
matrix that does not exist, which both fabricates a boundary the physics does not have and breaks
equivariance under the periodic roll used for augmentation later.

Move the sliders. The output shape is computed from the formula alone, before any tensor is built.

In [ ]:
# --- Interactive output shape calculator -------------------------------------
def sobel_x(k):
    # A vertical edge kernel of any odd side k, so every dropdown value draws a real feature map.
    c = np.arange(k) - (k - 1) / 2.0
    smooth = np.exp(-(c**2) / (2 * (k / 4.0)**2))      # separable: smooth along y
    return np.outer(smooth, c).astype(np.float32)      # antisymmetric along x

def shape_calculator(H=64, kernel=3, padding=1, stride=1, pad_mode="zeros"):
    H_out = (H + 2 * padding - kernel) // stride + 1
    print(f"n_out = floor((n + 2p - k) / s) + 1 = floor(({H} + 2x{padding} - {kernel})"
          f" / {stride}) + 1 = {H_out}")
    print(f"each output cell sees a {kernel}x{kernel} window of the input")
    print(f"a stack of three such layers, each followed by 2x2 pooling, would end at "
          f"{H_out // 8} x {H_out // 8}")

    demo = SMALL if H <= 12 else SYN[:H, :H]
    if pad_mode == "zeros":
        padded = np.pad(demo, padding, mode="constant")
    else:
        padded = np.pad(demo, padding, mode="wrap")

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
    axes[0].imshow(padded, cmap="gray", interpolation="nearest")
    axes[0].add_patch(Rectangle((padding - 0.5, padding - 0.5), demo.shape[1], demo.shape[0],
                                fill=False, ec=C_FIT, lw=2))
    n_show = 0
    for i in range(0, padded.shape[0] - kernel + 1, stride):
        for j in range(0, padded.shape[1] - kernel + 1, stride):
            if (i // stride + j // stride) % 3 == 0 and n_show < 220:
                axes[0].add_patch(Rectangle((j - 0.5, i - 0.5), kernel, kernel,
                                            fill=False, ec=C_DATA, lw=0.5, alpha=0.7))
                n_show += 1
    axes[0].set_title(f"{pad_mode} padding P = {padding}\norange box is the original image",
                      fontsize=9)

    out = conv2d_valid(padded, sobel_x(kernel))[::stride, ::stride]
    m = np.abs(out).max() + 1e-9
    axes[1].imshow(out, cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    axes[1].set_title(f"feature map, {out.shape[0]} x {out.shape[1]}\n"
                      f"vertical edge kernel, {kernel}x{kernel}", fontsize=9)
    for a in axes:
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
    plt.tight_layout(); plt.show()

interact(shape_calculator,
         H=Dropdown(options=[12, 32, 64], value=32),
         kernel=Dropdown(options=[3, 5, 7], value=3),
         padding=IntSlider(1, min=0, max=4, step=1, continuous_update=False),
         stride=IntSlider(1, min=1, max=4, step=1, continuous_update=False),
         pad_mode=Dropdown(options=["zeros", "wrap"], value="zeros"));


In [ ]:
# --- Check the formula against torch -----------------------------------------
t_in = torch.tensor(SYN).view(1, 1, 64, 64)
print(f"{'K':>3s} {'P':>3s} {'S':>3s} {'formula':>9s} {'torch':>9s}")
for (K_, P_, S_) in [(3, 0, 1), (3, 1, 1), (5, 2, 1), (3, 1, 2), (7, 3, 2), (3, 0, 3)]:
    formula = (64 + 2 * P_ - K_) // S_ + 1
    got = nn.Conv2d(1, 1, K_, padding=P_, stride=S_)(t_in).shape[-1]
    print(f"{K_:>3d} {P_:>3d} {S_:>3d} {formula:>9d} {got:>9d}   "
          f"{'ok' if formula == got else 'MISMATCH'}")


**What the table shows.** The formula and `nn.Conv2d` agree on every row, including the rows
where the division does not come out whole and the floor does the work. Being able to write
$n_{out}$ down without a computer is worth the two minutes it takes to learn, because a shape
mismatch between the last convolution block and the dense head is the most common error when
building one of these networks.

### 1f. Pooling

Pooling reduces the spatial size by summarising each window with a single number. For a window of
side $q$ and stride $q$, max pooling is

$$\boxed{\;O(i,j) \;=\; \max_{0 \,\le\, m,\, n \,<\, q}\; I\bigl(qi + m,\; qj + n\bigr)\;}$$

the largest of the $q^2$ input values in the window whose top-left corner sits at $(qi, qj)$.
Average pooling replaces the maximum by the mean over the same window. Both are fixed operations with
no parameters, and both follow the same output size formula as a convolution with $k = s = q$ and
$p = 0$, so $q = 2$ halves each side.

Pooling does two things. It cuts the amount of computation in the layers that follow, and it makes
the representation tolerant to small shifts: a feature that moves by one pixel usually lands in the
same pooling window and gives the same output.

The cell below pools a feature map, then repeats the whole thing on an input shifted by one pixel,
and measures how much the two results differ before and after pooling.

In [ ]:
# --- Max pooling, and shift tolerance ----------------------------------------
def maxpool(a, size=2):
    h, w = (a.shape[0] // size) * size, (a.shape[1] // size) * size
    return a[:h, :w].reshape(h // size, size, w // size, size).max(axis=(1, 3))

toy = np.array([[3, 1, 2, 8], [5, 6, 4, 2], [9, 1, 3, 7], [4, 8, 2, 5]], dtype=np.float32)
print("2x2 max pooling, stride 2, on a 4x4 input")
print(toy)
print("->")
print(maxpool(toy))
print()

f0 = conv2d_valid(SYN, K_EDGE_V)
f1 = conv2d_valid(np.roll(SYN, 1, axis=1), K_EDGE_V)      # input shifted by one pixel
p0, p1 = maxpool(np.abs(f0), 4), maxpool(np.abs(f1), 4)

d_raw  = np.abs(f0 - f1).mean() / (np.abs(f0).mean() + 1e-9)
d_pool = np.abs(p0 - p1).mean() / (np.abs(p0).mean() + 1e-9)
print(f"shift the input by one pixel:")
print(f"  mean relative change in the feature map      {d_raw:.4f}")
print(f"  mean relative change after 4x4 max pooling   {d_pool:.4f}")

fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.6))
for ax, (a, t) in zip(axes, [(f0, "feature map"), (f1, "same, input shifted 1 px"),
                             (p0, "pooled"), (p1, "pooled, shifted")]):
    m = np.abs(a).max()
    ax.imshow(a, cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    ax.set_title(f"{t}\n{a.shape}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()


### 1g. Weight sharing, and the parameter count

This is the argument that makes convolution worth using on images, and it is arithmetic, not
opinion.

Notebook 2 gave the dense layer count. For $n_{in}$ inputs and $n_{out}$ outputs,

$$P_{\text{dense}} \;=\; n_{in}\,n_{out} + n_{out}$$

A convolution layer holds one $k \times k$ kernel for every pair of input and output channel, plus
one bias per output channel:

$$\boxed{\;P_{\text{conv}} \;=\; k^2\, c_{in}\, c_{out} + c_{out}\;}$$

$k$ is the kernel side, $c_{in}$ the number of input channels and $c_{out}$ the number of filters.
The image size does not appear. That is the whole difference: the dense count is set by how many
pixels there are, the convolution count by how large a pattern you are looking for and how many
different patterns you want.

Put the two side by side on this dataset. A dense layer from the flattened 64x64 image to 256 hidden
units has $n_{in} = 4096$, so $P_{\text{dense}} = 4096 \times 256 + 256$. A first convolution layer
with $k = 3$, $c_{in} = 1$ and $c_{out} = 16$ has $P_{\text{conv}} = 9 \times 1 \times 16 + 16$. The
cell below evaluates both and prints the ratio, then repeats it at other image sizes.

In [ ]:
# --- Parameters: dense versus convolutional ----------------------------------
sizes = np.array([32, 64, 96, 128, 192, 256])
H_units, F_filters, K_size, C_in = 256, 16, 3, 1

dense_p = sizes**2 * H_units + H_units                              # n_in*n_out + n_out
conv_p  = np.full_like(sizes, K_size**2 * C_in * F_filters + F_filters)   # k^2*c_in*c_out + c_out

print(f"P_dense = n_in * n_out + n_out   with n_out = {H_units}")
print(f"P_conv  = k^2 * c_in * c_out + c_out = {K_size}^2 x {C_in} x {F_filters}"
      f" + {F_filters} = {conv_p[0]}")
print()
print(f"{'image':>7s} {'dense (256 units)':>19s} {'conv (16 filters 3x3)':>23s} {'ratio':>9s}")
for s, d, c in zip(sizes, dense_p, conv_p):
    print(f"{s:>4d}^2 {d:>19,d} {c:>23,d} {d/c:>9,.0f}x")

fig, ax = plt.subplots(figsize=(6.8, 4))
ax.plot(sizes, dense_p, "-o", color=C_BAD, lw=2, label=f"dense layer, {H_units} units")
ax.plot(sizes, conv_p, "-s", color=C_DATA, lw=2, label=f"conv layer, {F_filters} filters 3x3")
ax.axvline(64, color="k", ls=":", lw=1.2)
ax.text(66, 3e2, "this dataset", fontsize=8)
ax.set_yscale("log"); ax.set_xlabel("image side (pixels)"); ax.set_ylabel("parameters")
ax.set_title("weight sharing removes the dependence on image size", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


The conv line is flat. That is weight sharing: the number of parameters is set by the size
of the pattern you are looking for, not by the size of the picture you are looking in.

There is a second, less obvious gain. A dense layer has to learn what a fibre boundary looks like
separately at every location, from whatever examples happen to put a boundary there. A convolutional
layer sees every location as another example of the same filter, so 1470 images give it far more
effective training data per parameter.

### 1h. A forward pass through an untrained CNN

Three convolution blocks, each one convolution, a ReLU and a 2x2 max pool. The image shrinks by half
at each block and the number of channels grows. The weights here are random, so the feature maps
mean nothing yet. The shapes are the point.

In [ ]:
# --- Shapes through an untrained network -------------------------------------
torch.manual_seed(SEED)
demo_net = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.MaxPool2d(2),
)

h = torch.tensor(SYN).view(1, 1, 64, 64)
stages = [("input", h[0, 0].numpy()[None])]
print(f"{'layer':22s} {'output shape':>20s} {'parameters':>12s}")
print(f"{'input':22s} {str(tuple(h.shape)):>20s} {0:>12d}")
for layer in demo_net:
    h = layer(h)
    npar = sum(p.numel() for p in layer.parameters())
    print(f"{layer.__class__.__name__:22s} {str(tuple(h.shape)):>20s} {npar:>12,d}")
    if isinstance(layer, nn.MaxPool2d):
        stages.append((f"after pool, {tuple(h.shape)[1:]}", h[0].detach().numpy()))

fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for r, (name, maps) in enumerate(stages):
    for c in range(4):
        ax = axes[r, c]
        if c < maps.shape[0]:
            ax.imshow(maps[c], cmap="viridis", interpolation="nearest")
        else:
            ax.axis("off")
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if c == 0:
            ax.set_ylabel(name, fontsize=8)
plt.suptitle("first four channels at each stage, untrained weights", y=0.995, fontsize=11)
plt.tight_layout(); plt.show()


---

# Part 2 - A CNN on the microstructures: average stiffness

The network used for the rest of this notebook is deliberately ordinary. Three convolution blocks,
each one 3x3 convolution, a ReLU and a 2x2 max pool, then a small dense head.

| Block | Operation | Output |
|---|---|---|
| input | | 1 x 64 x 64 |
| 1 | conv 3x3, 16 filters, ReLU, pool | 16 x 32 x 32 |
| 2 | conv 3x3, 32 filters, ReLU, pool | 32 x 16 x 16 |
| 3 | conv 3x3, 64 filters, ReLU, pool | 64 x 8 x 8 |
| head | flatten, dense 64, ReLU, dense 1 | 1 |

The padding is circular, not zero. The microstructures are periodic cells, so a fibre that leaves
the right edge re-enters on the left, and circular padding is the only choice consistent with the
physics. Zero padding would tell the network there is empty matrix outside the cell, which is false.

In [ ]:
# --- The network --------------------------------------------------------------
def make_cnn(widths=(16, 32, 64), head=64, in_size=64, pad_mode="circular", seed=SEED):
    torch.manual_seed(seed)
    layers, c = [], 1
    for w in widths:
        layers += [nn.Conv2d(c, w, 3, padding=1, padding_mode=pad_mode),
                   nn.ReLU(), nn.MaxPool2d(2)]
        c = w
    s = in_size // (2 ** len(widths))
    layers += [nn.Flatten(), nn.Linear(c * s * s, head), nn.ReLU(), nn.Linear(head, 1)]
    return nn.Sequential(*layers)

cnn_demo = make_cnn()
N_PARAM_CNN = sum(p.numel() for p in cnn_demo.parameters())

conv_par = sum(p.numel() for m in cnn_demo if isinstance(m, nn.Conv2d) for p in m.parameters())
dense_par = N_PARAM_CNN - conv_par

# the Notebook 2 flattened MLP, for comparison
mlp_flat_sizes = [4096, 256, 64, 1]
N_PARAM_MLP = sum(a * b + b for a, b in zip(mlp_flat_sizes[:-1], mlp_flat_sizes[1:]))

print(cnn_demo)
print()
print(f"CNN total parameters      {N_PARAM_CNN:>10,d}")
print(f"  in the convolutions     {conv_par:>10,d}   ({100*conv_par/N_PARAM_CNN:.1f}%)")
print(f"  in the dense head       {dense_par:>10,d}")
print(f"NB2 flattened MLP         {N_PARAM_MLP:>10,d}")


### The training budget

Everything below is sized for two CPU cores, which is what a free Colab CPU runtime gives you. The
epoch counts are set once here so they are easy to find and change. If a GPU is available the counts
are doubled, since the same run then costs a fraction of the time.

In [ ]:
# --- Budget -------------------------------------------------------------------
EPOCHS_MEAN = 20 if DEVICE == "cpu" else 40     # target E_mean, Part 2
EPOCHS_DE   = 40 if DEVICE == "cpu" else 80     # target dE, Part 3

print(f"E_mean: {EPOCHS_MEAN} epochs   dE: {EPOCHS_DE} epochs   on {DEVICE}")
print(f"Four CNN runs follow: {EPOCHS_MEAN} epochs on E_mean, {EPOCHS_MEAN} on E_mean with the")
print(f"pixels shuffled, {EPOCHS_DE} on dE, and {EPOCHS_DE} on dE with the pixels shuffled.")
print(f"That is {2*EPOCHS_MEAN + 2*EPOCHS_DE} epochs in total. Each cell prints its own elapsed")
print("time and its own cost per epoch, so there is no need to take an estimate on trust.")


Most of the parameters are in the dense head, not in the convolutions. The three convolution
blocks that do all the pattern detection cost about twenty thousand numbers between them. That is
weight sharing paying for itself.

### Augmentation, and which symmetries are real

The training set has 1470 images. Augmentation manufactures more by applying transformations that
leave the target unchanged. The transformations have to be chosen from the physics, not from a list
of defaults.

- **Periodic roll.** Shifting the image cyclically in x or y gives the same periodic cell with a
  different origin. The homogenised properties are identical. This is an exact symmetry of the data
  and it is free.
- **Flips.** Mirroring in x or in y maps the cell to a valid cell and leaves $E_{22}$ and $E_{33}$
  each unchanged, so both $E_{mean}$ and $\Delta E$ survive.
- **Rotation by 90 degrees.** This one is not neutral. It exchanges the two in-plane axes, so
  $E_{22}$ and $E_{33}$ swap, $E_{mean}$ is unchanged but $\Delta E$ changes sign. It can be used,
  but only if the target is transformed with the image. It is left out below to keep the augmentation
  code short.

Applying a transformation that is not a symmetry of the target is one of the more common ways to
quietly poison a training run.

In [ ]:
# --- Augmentation -------------------------------------------------------------
def augment(xb, rng=np.random):
    sy, sx = rng.randint(0, xb.shape[-2]), rng.randint(0, xb.shape[-1])
    xb = torch.roll(xb, shifts=(int(sy), int(sx)), dims=(2, 3))    # periodic roll
    if rng.rand() < 0.5: xb = torch.flip(xb, dims=[3])             # mirror in x
    if rng.rand() < 0.5: xb = torch.flip(xb, dims=[2])             # mirror in y
    return xb

np.random.seed(SEED)
ex = torch.tensor(X_IMG[3]).view(1, 1, 64, 64)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.7))
axes[0].imshow(ex[0, 0], cmap="gray", interpolation="nearest")
axes[0].set_title("original", fontsize=9)
for k in range(1, 6):
    a = augment(ex.clone())
    axes[k].imshow(a[0, 0], cmap="gray", interpolation="nearest")
    axes[k].set_title(f"augmented {k}\n$V_f$ = {a.mean():.3f}", fontsize=9)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle("periodic roll plus flips: the same cell, a different origin", y=1.06, fontsize=10)
plt.tight_layout(); plt.show()


### Training

One helper trains every model below. The target is standardised using the training set only, which
keeps the learning rate meaningful across two targets with very different magnitudes. The test
$R^2$ is recorded every few epochs so the curve can be plotted rather than a single final number.

In [ ]:
# --- Training helper ----------------------------------------------------------
def train_cnn(model, Xtr, ytr, Xte, yte, epochs=40, bs=32, lr=1e-3,
              augmented=True, eval_every=2, seed=SEED, tag=""):
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)
    model = model.to(DEVICE)

    mu, sd = float(ytr.mean()), float(ytr.std())          # target standardisation, train only
    A = torch.tensor(Xtr).unsqueeze(1).to(DEVICE)
    B = torch.tensor((ytr - mu) / sd).view(-1, 1).to(DEVICE)
    P = torch.tensor(Xte).unsqueeze(1).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    n = len(A)
    hist_ep, hist_r2, hist_loss = [], [], []

    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        run = 0.0
        for i in range(0, n, bs):
            j  = perm[i:i + bs]
            xb = augment(A[j].clone(), rng) if augmented else A[j]
            opt.zero_grad()
            l = lossf(model(xb), B[j])
            l.backward(); opt.step()
            run += l.item() * len(j)
        hist_loss.append(run / n)
        if ep % eval_every == eval_every - 1 or ep == epochs - 1:
            model.eval()
            with torch.no_grad():
                pr = model(P).cpu().numpy().ravel() * sd + mu
            hist_ep.append(ep + 1); hist_r2.append(r2_score(yte, pr))
    elapsed = time.time() - t0

    model.eval()
    with torch.no_grad():
        pred = model(P).cpu().numpy().ravel() * sd + mu
    r2 = r2_score(yte, pred)
    print(f"{tag}{epochs} epochs, augmented={augmented}: {elapsed:.1f} s "
          f"({elapsed/epochs:.2f} s per epoch), test R2 = {r2:.4f}")
    return dict(r2=r2, elapsed=elapsed, pred=pred, ep=hist_ep, r2_hist=hist_r2,
                loss=hist_loss, model=model)


In [ ]:
# --- Train the CNN on E_mean --------------------------------------------------
idx = np.arange(len(clean))
itr, ite = train_test_split(idx, test_size=0.25, random_state=SEED)
Xtr_img, Xte_img = X_IMG[itr], X_IMG[ite]
ym_tr, ym_te = y_mean[itr], y_mean[ite]
yd_tr, yd_te = y_dE[itr],  y_dE[ite]

print(f"train {len(itr)}   test {len(ite)}")
res_mean = train_cnn(make_cnn(), Xtr_img, ym_tr, Xte_img, ym_te,
                     epochs=EPOCHS_MEAN, tag="CNN on E_mean, ")
R2_CNN_MEAN = res_mean["r2"]


In [ ]:
# --- How it trained -----------------------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(res_mean["loss"], color=C_DATA, lw=1.8)
a1.set_yscale("log"); a1.set_xlabel("epoch"); a1.set_ylabel("training MSE (standardised target)")
a1.set_title("training loss", fontsize=10)

a2.scatter(ym_te, res_mean["pred"], s=12, alpha=0.5, color=C_DATA)
lim = [ym_te.min() - 0.4, ym_te.max() + 0.4]
a2.plot(lim, lim, "k--", lw=1)
a2.set_xlim(lim); a2.set_ylim(lim)
a2.set_xlabel("true $E_{mean}$ (GPa)"); a2.set_ylabel("predicted $E_{mean}$ (GPa)")
a2.set_title(f"CNN on the raw image, test $R^2$ = {R2_CNN_MEAN:.4f}", fontsize=10)
plt.tight_layout(); plt.show()


### The baselines, on the same split

Two numbers are needed for context: the Notebook 1 linear model on volume fraction, and the
Notebook 2 fully connected network on the flattened image. The flattened network below is the same
architecture, the same epoch count, the same seed, the same split and the same target standardisation
as the one in Notebook 2, so it reproduces that notebook's printed score rather than a new one.

In [ ]:
# --- Notebook 1 and Notebook 2 baselines, recomputed on this split -----------
vf_all = clean["vf"].values.reshape(-1, 1)
lin_mean = LinearRegression().fit(vf_all[itr], ym_tr)
R2_LIN_MEAN = r2_score(ym_te, lin_mean.predict(vf_all[ite]))

X_FLAT = X_IMG.reshape(len(clean), -1)

def train_flat_mlp(Xtr_, ytr_, Xte_, yte_, epochs=12, bs=32, lr=1e-3, seed=SEED, tag=""):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(4096, 256), nn.ReLU(),
                      nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
    mu, sd = float(ytr_.mean()), float(ytr_.std())
    A = torch.tensor(Xtr_); B = torch.tensor((ytr_ - mu) / sd).view(-1, 1)
    P = torch.tensor(Xte_)
    opt = torch.optim.Adam(m.parameters(), lr=lr); lf = nn.MSELoss()
    t0 = time.time(); n = len(A)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            j = perm[i:i + bs]
            opt.zero_grad(); lf(m(A[j]), B[j]).backward(); opt.step()
    el = time.time() - t0
    with torch.no_grad():
        pred = m(P).numpy().ravel() * sd + mu
    r2 = r2_score(yte_, pred)
    print(f"{tag}flattened MLP, {epochs} epochs: {el:.1f} s, test R2 = {r2:.4f}")
    return r2, el

R2_MLP_MEAN, el_mlp_mean = train_flat_mlp(X_FLAT[itr], ym_tr, X_FLAT[ite], ym_te,
                                          tag="E_mean, ")
print(f"\nlinear in Vf only: test R2 = {R2_LIN_MEAN:.4f}")


### The permutation test

Notebook 2 applied one fixed random permutation to the 4096 pixel positions of every image, training
and test alike, and the fully connected network scored the same on the shuffled images as on the real
ones. That is a property of the architecture: permuting the input and permuting the columns of the
first weight matrix the same way gives an identical network.

A convolution has no such freedom. Its parameters are tied to spatial offsets, so a shuffle
destroys every relationship a 3x3 filter can express.

Run the test on both models, on $E_{mean}$, and predict the outcome before looking.

In [ ]:
# --- The same fixed permutation as Notebook 2 --------------------------------
perm_pix = np.random.default_rng(1).permutation(4096)
X_SHUF = X_FLAT[:, perm_pix].reshape(-1, 64, 64)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
for k, i in enumerate([0, 5]):
    axes[2*k].imshow(X_IMG[i], cmap="gray", interpolation="nearest")
    axes[2*k].set_title(f"microstructure\n$V_f$ = {X_IMG[i].mean():.3f}", fontsize=9)
    axes[2*k+1].imshow(X_SHUF[i], cmap="gray", interpolation="nearest")
    axes[2*k+1].set_title(f"pixels shuffled\n$V_f$ = {X_SHUF[i].mean():.3f}", fontsize=9)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle("one fixed permutation, applied to every image in the dataset", y=1.05)
plt.tight_layout(); plt.show()


In [ ]:
# --- Retrain both models on the shuffled pixels ------------------------------
res_mean_shuf = train_cnn(make_cnn(), X_SHUF[itr], ym_tr, X_SHUF[ite], ym_te,
                          epochs=EPOCHS_MEAN, tag="CNN on shuffled pixels, ")
R2_CNN_MEAN_SHUF = res_mean_shuf["r2"]

R2_MLP_MEAN_SHUF, _ = train_flat_mlp(X_FLAT[:, perm_pix][itr], ym_tr,
                                     X_FLAT[:, perm_pix][ite], ym_te,
                                     tag="E_mean shuffled, ")

print()
print(f"{'model':28s} {'real images':>12s} {'shuffled':>10s} {'change':>9s}")
print(f"{'flattened MLP (NB2)':28s} {R2_MLP_MEAN:12.4f} {R2_MLP_MEAN_SHUF:10.4f} "
      f"{R2_MLP_MEAN_SHUF - R2_MLP_MEAN:+9.4f}")
print(f"{'CNN':28s} {R2_CNN_MEAN:12.4f} {R2_CNN_MEAN_SHUF:10.4f} "
      f"{R2_CNN_MEAN_SHUF - R2_CNN_MEAN:+9.4f}")


In [ ]:
# --- The same comparison as a picture ----------------------------------------
fig, ax = plt.subplots(figsize=(6.6, 4))
x = np.arange(2); w = 0.36
ax.bar(x - w/2, [R2_MLP_MEAN, R2_CNN_MEAN], w, color=C_DATA, label="real microstructures")
ax.bar(x + w/2, [R2_MLP_MEAN_SHUF, R2_CNN_MEAN_SHUF], w, color=C_BAD, label="pixels shuffled")
ax.set_xticks(x); ax.set_xticklabels(["flattened MLP", "CNN"])
ax.set_ylabel("test $R^2$ on $E_{mean}$")
ax.set_ylim(min(0, R2_CNN_MEAN_SHUF - 0.1), 1.05)
ax.axhline(0, color="k", lw=0.8)
ax.legend(fontsize=8, loc="lower left")
ax.set_title("the permutation test on $E_{mean}$: no effect on either model", fontsize=10)
plt.tight_layout(); plt.show()


Neither model moved. That is not the answer the argument wanted, and it is worth more than
the answer the argument wanted.

The permutation destroys the geometry, but it does not change the number of white pixels, and
$E_{mean}$ is very nearly a function of that number alone. The linear fit on volume fraction above
already scores $R^2 = 0.95$ with two parameters. A convolutional network can count white pixels on a
shuffled image as easily as on a real one: a filter with all-positive weights followed by pooling is
a blurred pixel count.

So the permutation test is not a property of the architecture alone. It only bites on a target that
needs geometry. Part 3 has one.

---

# Part 3 - Anisotropy, and an honest result

$\Delta E = E_{22} - E_{33}$ is where the geometry lives. Notebook 1 found $R^2 = 0.0000$ for
volume fraction. Notebook 2 found that a short list of physically motivated descriptors recovers
most of it.

Now train the same CNN on the same split, on $\Delta E$, with the same augmentation. The
descriptors were computed from these same 64x64 images and from nothing else, so the raw image the
CNN receives contains strictly more information than the descriptor vector. The question is whether
the network can get at it inside a teaching time budget.

In [ ]:
# --- Train the CNN on the anisotropy -----------------------------------------
res_dE = train_cnn(make_cnn(), Xtr_img, yd_tr, Xte_img, yd_te,
                   epochs=EPOCHS_DE, tag="CNN on dE, ")
R2_CNN_DE = res_dE["r2"]


In [ ]:
# --- The descriptor baselines, on the same split -----------------------------
# 14 named descriptors, the baseline Notebook 2 teaches
t0 = time.time()
gb_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_DESC[itr], yd_tr)
R2_DESC_DE = r2_score(yd_te, gb_dE.predict(X_DESC[ite]))
el_desc = time.time() - t0
gb_mean = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_DESC[itr], ym_tr)
R2_DESC_MEAN = r2_score(ym_te, gb_mean.predict(X_DESC[ite]))

# the full research bank of 30, same model, same split
t0 = time.time()
gb30_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_D30[itr], yd_tr)
R2_D30_DE = r2_score(yd_te, gb30_dE.predict(X_D30[ite]))
el_d30 = time.time() - t0
gb30_mean = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_D30[itr], ym_tr)
R2_D30_MEAN = r2_score(ym_te, gb30_mean.predict(X_D30[ite]))

lin_dE = LinearRegression().fit(vf_all[itr], yd_tr)
R2_LIN_DE = r2_score(yd_te, lin_dE.predict(vf_all[ite]))

R2_MLP_DE, _ = train_flat_mlp(X_FLAT[itr], yd_tr, X_FLAT[ite], yd_te, tag="dE, ")

print(f"\n{'representation':34s} {'dim':>4s} {'fit s':>7s} {'R2 on dE':>9s}")
print(f"{'14 named descriptors':34s} {X_DESC.shape[1]:4d} {el_desc:7.1f} {R2_DESC_DE:9.4f}")
print(f"{'full bank, 30 descriptors':34s} {X_D30.shape[1]:4d} {el_d30:7.1f} {R2_D30_DE:9.4f}")
print(f"{'volume fraction only, linear':34s} {1:4d} {0.0:7.1f} {R2_LIN_DE:9.4f}")
print(f"{'raw image 64x64, CNN':34s} {64*64:4d} {res_dE['elapsed']:7.1f} {R2_CNN_DE:9.4f}")

# which of the 14 carries the anisotropy
from sklearn.inspection import permutation_importance
pi = permutation_importance(gb_dE, X_DESC[ite], yd_te, n_repeats=5, random_state=SEED)
ord_pi = np.argsort(pi.importances_mean)[::-1]
TOP_DESC = DESC_NAMES[ord_pi[0]]
LAG_TOP  = int(TOP_DESC.split("_")[-1])
print("\npermutation importance on the 14 named descriptors, target dE")
for k in ord_pi[:4]:
    print(f"   {DESC_NAMES[k]:20s} {pi.importances_mean[k]:7.4f}   {DESC_MEANINGS[k]}")
print(f"\ndominant descriptor {TOP_DESC}, a directional statistic at a lag of {LAG_TOP} pixels")

R2_PUB_DE = 0.942      # published tuned directional CNN on this data, quoted for reference only
print(f"published tuned research CNN on dE, for reference only: {R2_PUB_DE:.3f}")


In [ ]:
# --- Parity plot and the test curve ------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))

a1.plot(res_dE["ep"], res_dE["r2_hist"], "-o", ms=3, color=C_DATA, label="CNN, test $R^2$")
a1.axhline(R2_DESC_DE, color=C_ALT, ls="--", lw=1.8,
           label=f"14 named descriptors, {R2_DESC_DE:.3f}")
a1.axhline(R2_D30_DE, color=C_ALT, ls="-.", lw=1.4,
           label=f"full bank of 30, {R2_D30_DE:.3f}")
a1.axhline(R2_PUB_DE, color=C_FIT, ls=":", lw=1.8,
           label=f"tuned research CNN, published, {R2_PUB_DE:.3f}")
a1.set_xlabel("epoch"); a1.set_ylabel("test $R^2$ on $\\Delta E$")
a1.set_ylim(-0.1, 1.0); a1.legend(fontsize=8, loc="lower right")
a1.set_title("the CNN against its competition", fontsize=10)

a2.scatter(yd_te, res_dE["pred"], s=12, alpha=0.5, color=C_DATA)
lim = [yd_te.min() - 0.2, yd_te.max() + 0.2]
a2.plot(lim, lim, "k--", lw=1)
a2.set_xlim(lim); a2.set_ylim(lim)
a2.set_xlabel("true $\\Delta E$ (GPa)"); a2.set_ylabel("predicted $\\Delta E$ (GPa)")
a2.set_title(f"CNN on $\\Delta E$, test $R^2$ = {R2_CNN_DE:.4f}", fontsize=10)
plt.tight_layout(); plt.show()


**What the two panels show, and it is the central result of this notebook.** Left: the blue
curve is the CNN's test $R^2$ on $\Delta E$ as training proceeds. The green dashed line is the 14
named descriptors with gradient boosting, on the same split, and the dash-dotted line above it is
the full bank of 30. The CNN curve ends below both. The dotted line higher up is the published tuned
architecture, quoted for scale.

**The CNN loses.** It was given the full 64x64 image. The descriptors were computed from those same
64x64 images, so the image contains strictly more information than the descriptor vector, and the
comparison is like for like at one resolution. The CNN was given roughly a quarter of a million
parameters and the longest training run in this notebook. It still scores below a gradient boosting
model fitted to fourteen numbers that a person chose, at a fraction of the cost, as the printed
timings show. All the scores are printed above. Compare them and take the comparison at face value.

Right: the parity plot for the CNN. The cloud is tilted towards the diagonal, so the network has
learned something real about anisotropy, and it is wide, so a substantial part of the variance is
still unexplained. The section after the permutation test explains why.

### The permutation test, on the target that needs geometry

Now repeat the Part 2 experiment on $\Delta E$. The same fixed permutation, the same architecture,
the same epoch budget, the same split. The only difference is that this target cannot be reached by
counting pixels.

In [ ]:
# --- CNN on shuffled pixels, target dE ---------------------------------------
res_dE_shuf = train_cnn(make_cnn(), X_SHUF[itr], yd_tr, X_SHUF[ite], yd_te,
                        epochs=EPOCHS_DE, tag="CNN on dE, shuffled pixels, ")
R2_CNN_DE_SHUF = res_dE_shuf["r2"]

print()
print(f"{'model and target':32s} {'real':>9s} {'shuffled':>10s} {'change':>9s}")
print(f"{'flattened MLP, E_mean':32s} {R2_MLP_MEAN:9.4f} {R2_MLP_MEAN_SHUF:10.4f} "
      f"{R2_MLP_MEAN_SHUF - R2_MLP_MEAN:+9.4f}")
print(f"{'CNN, E_mean':32s} {R2_CNN_MEAN:9.4f} {R2_CNN_MEAN_SHUF:10.4f} "
      f"{R2_CNN_MEAN_SHUF - R2_CNN_MEAN:+9.4f}")
print(f"{'CNN, dE':32s} {R2_CNN_DE:9.4f} {R2_CNN_DE_SHUF:10.4f} "
      f"{R2_CNN_DE_SHUF - R2_CNN_DE:+9.4f}")


In [ ]:
# --- The permutation test that separates the two architectures ---------------
fig, ax = plt.subplots(figsize=(7.2, 4))
x = np.arange(3); w = 0.36
real = [R2_MLP_MEAN, R2_CNN_MEAN, R2_CNN_DE]
shuf = [R2_MLP_MEAN_SHUF, R2_CNN_MEAN_SHUF, R2_CNN_DE_SHUF]
ax.bar(x - w/2, real, w, color=C_DATA, label="real microstructures")
ax.bar(x + w/2, shuf, w, color=C_BAD,  label="pixels shuffled")
ax.set_xticks(x)
ax.set_xticklabels(["flattened MLP\n$E_{mean}$", "CNN\n$E_{mean}$", "CNN\n$\\Delta E$"])
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("test $R^2$"); ax.set_ylim(min(-0.1, min(shuf) - 0.1), 1.08)
ax.legend(fontsize=8, loc="lower left")
ax.set_title("shuffling the pixels costs nothing until the target needs geometry", fontsize=10)
plt.tight_layout(); plt.show()


On $\Delta E$ the CNN collapses when the pixels are shuffled, while on $E_{mean}$ it did not.
The architecture is the same in both runs, so the difference is in what the target demands.

Put the two halves together and the claim is now properly supported. The convolutional network is
using the spatial arrangement of the fibres, because removing that arrangement removes its score.
The fully connected network of Notebook 2 never used it, because removing it changed nothing.

### Read this result carefully

The CNN did not beat the descriptors. Three things are going on, and none of them is a bug.

**The descriptors already contain the answer.** The permutation importance printed above puts one
descriptor far above the rest: the difference between the periodic two-point correlation along x and
along y, at a lag of a few fibre radii. A convolutional network can in principle compute something
like it, but only by learning a directional statistic over that range out of stacked 3x3 filters,
from 1470 examples. The descriptor was handed to the other model for free.

**The signal is small and the data are few.** $\Delta E$ ranges over about 3 GPa around zero and is
the difference of two nearly equal numbers. There are 1102 training images. Fitting a quarter of a
million parameters to that is not a comfortable position, which is why the test curve above moves
around as much as it does.

**The architecture is generic.** Nothing in it knows that the target is a difference between two
directions. The published study on this data reached the score printed as `R2_PUB_DE` for the same
quantity, with a tuned directional architecture and a training budget far beyond what fits in a
lecture. That number is the gap between a quick CNN and a considered one, and it is quoted here as
context, not as something this notebook reaches.

The lesson is not that CNNs are bad. It is that a CNN is a hypothesis about structure, and a
physically motivated feature set is a much stronger hypothesis than most people expect. Try both,
and quote both honestly.

### The representation ladder

Before the summary figure, one more rung. Notebook 2 introduced the two-point correlation
$S_2$ as a full statistical description of the microstructure, and the 2019 route compresses it with
PCA before regression. The next cell computes that representation here, on the same 64x64 images and
the same split, so it can sit in the same figure as everything else.

In [ ]:
# --- Two-point correlation + PCA-50, the 2019 route, on this split -----------
t0 = time.time()

def s2_map(im):
    v = im.astype(np.float64)
    f = np.fft.rfft2(v)
    return np.fft.fftshift(np.fft.irfft2(f * np.conj(f), s=v.shape) / v.size)

S2_FLAT = np.array([s2_map(im).ravel() for im in X_IMG])
el_s2 = time.time() - t0

from sklearn.decomposition import PCA
pca_s2 = PCA(n_components=50, svd_solver="full", random_state=SEED).fit(S2_FLAT[itr])
S2_PC = pca_s2.transform(S2_FLAT)

gb_s2_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(S2_PC[itr], yd_tr)
R2_S2_DE = r2_score(yd_te, gb_s2_dE.predict(S2_PC[ite]))
gb_s2_m  = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(S2_PC[itr], ym_tr)
R2_S2_MEAN = r2_score(ym_te, gb_s2_m.predict(S2_PC[ite]))

# composition alone: volume fraction and diameter, same model
X_COMP = clean[["vol_frac", "diameter"]].values.astype(np.float32)
gb_c_dE = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_COMP[itr], yd_tr)
R2_COMP_DE = r2_score(yd_te, gb_c_dE.predict(X_COMP[ite]))
gb_c_m  = HistGradientBoostingRegressor(random_state=SEED, max_iter=400).fit(X_COMP[itr], ym_tr)
R2_COMP_MEAN = r2_score(ym_te, gb_c_m.predict(X_COMP[ite]))

print(f"S2 for {len(X_IMG)} images: {el_s2:.1f} s, one map is {S2_FLAT.shape[1]} numbers")
print(f"PCA-50 keeps {pca_s2.explained_variance_ratio_.sum():.4f} of the variance of S2")
print(f"S2 + PCA-50      test R2   E_mean {R2_S2_MEAN:.4f}   dE {R2_S2_DE:.4f}")
print(f"composition      test R2   E_mean {R2_COMP_MEAN:.4f}   dE {R2_COMP_DE:.4f}")


In [ ]:
# --- The summary figure for the first day ------------------------------------
LADDER = [
    ("composition\n$V_f$ and fibre diameter", "composition, Vf and diameter",
     2, R2_COMP_MEAN, R2_COMP_DE, "#8C8C8C"),
    ("two-point correlation $S_2$\ncompressed to 50 PCs", "S2 + PCA-50",
     50, R2_S2_MEAN, R2_S2_DE, C_DATA),
    ("14 named descriptors\nchosen for the physics", "14 named descriptors",
     14, R2_DESC_MEAN, R2_DESC_DE, C_ALT),
    ("full descriptor bank", "full descriptor bank",
     30, R2_D30_MEAN, R2_D30_DE, "#2F6B43"),
    ("raw image 64x64\nCNN trained in this notebook", "raw image, CNN in this notebook",
     4096, R2_CNN_MEAN, R2_CNN_DE, C_FIT),
]
labels = [f"{r[0]}  ({r[2]})" for r in LADDER]
ypos = np.arange(len(LADDER))[::-1]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True)
for ax, col, title in [(a1, 3, "$E_{mean}$, average stiffness"),
                       (a2, 4, "$\\Delta E$, anisotropy")]:
    vals = [r[col] for r in LADDER]
    ax.barh(ypos, vals, color=[r[5] for r in LADDER], height=0.62)
    for yv, v in zip(ypos, vals):
        ax.text(v + 0.02 if v > 0 else 0.02, yv, f"{v:.3f}", va="center", fontsize=9.5)
    ax.axvline(0, color="k", lw=0.9)
    ax.set_xlim(min(-0.15, min(vals) - 0.06), 1.22)
    ax.set_xlabel("test $R^2$"); ax.set_title(title, fontsize=11)
    ax.grid(axis="y", alpha=0)

a2.axvline(R2_PUB_DE, color=C_BAD, ls=":", lw=2.0)
a2.text(R2_PUB_DE - 0.03, ypos[0] + 0.62, f"tuned research CNN\npublished, {R2_PUB_DE:.3f}",
        fontsize=8.5, color=C_BAD, ha="right", va="top")
a2.set_ylim(ypos[-1] - 0.6, ypos[0] + 1.1)
a1.set_yticks(ypos); a1.set_yticklabels(labels, fontsize=9.5)
plt.suptitle("what the input representation is worth, same split, same 64x64 images\n"
             "bracketed number is the dimension of the representation", y=1.03, fontsize=11)
plt.tight_layout(); plt.show()

print(f"{'input representation':34s} {'dim':>5s} {'E_mean':>9s} {'dE':>9s}")
for _, nm, d_, a, b, _c in LADDER:
    print(f"{nm:34s} {d_:5d} {a:9.4f} {b:9.4f}")
print(f"{'published tuned CNN (reference)':34s} {'-':>5s} {'-':>9s} {R2_PUB_DE:9.4f}")


Two different stories in one figure, and this is the summary of the first day.

On the left every representation works, because average stiffness is close to a function of the
volume fraction and every model can count white pixels. Read the left panel and you would conclude
that the choice of representation does not matter.

On the right it decides everything. Composition carries none of the anisotropy. The two-point
correlation compressed to 50 components recovers a good part of it. Fourteen named descriptors do
better than that with a third of the dimension, and the full bank of 30 does better again. The CNN
of this notebook sits below both descriptor rows, and the printed table gives the numbers.

The ladder is labelled by what goes into the model, not by which model it is, because every row
except the CNN uses the same gradient boosting regressor. The representation is the variable.

### The cost side of the comparison

$R^2$ is not the only axis. The descriptor route needs a person to design descriptors once, then
costs very little to fit, as the printed timings show. The CNN route needs no feature design but costs minutes to train here, and
would cost hours at research settings. Which trade is right depends on how many new problems you
expect to face and how much physics you already know.

---

# Part 4 - What the CNN learned

The network trained on $\Delta E$ is available as `res_dE["model"]`. Three views of it follow, in
increasing order of how easy they are to over-interpret.

### The first layer filters

Sixteen 3x3 kernels, learned rather than written down. Read them the way Part 1 read the fixed
kernels: the sum of the entries says whether the filter responds to average level or only to
change, and the arrangement of signs says which direction of change it prefers.

In [ ]:
# --- First layer filters ------------------------------------------------------
model_dE = res_dE["model"].cpu()
W1 = model_dE[0].weight.detach().numpy()[:, 0]          # (16, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
m = np.abs(W1).max()
for k, ax in enumerate(axes.ravel()):
    ax.imshow(W1[k], cmap="coolwarm", vmin=-m, vmax=m, interpolation="nearest")
    ax.set_title(f"sum {W1[k].sum():+.2f}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle("learned first layer filters", y=1.02, fontsize=11)
plt.tight_layout(); plt.show()

# how directional is each filter: horizontal minus vertical difference energy
dx = np.abs(np.diff(W1, axis=2)).sum(axis=(1, 2))
dy = np.abs(np.diff(W1, axis=1)).sum(axis=(1, 2))
print(f"{'filter':>7s} {'sum':>8s} {'x-variation':>12s} {'y-variation':>12s}")
for k in np.argsort(-(np.abs(dx - dy)))[:6]:
    print(f"{k:>7d} {W1[k].sum():>8.3f} {dx[k]:>12.3f} {dy[k]:>12.3f}")
print("\nFilters with x-variation far from y-variation are directional, which is what a target")
print("built from the difference between two directions would be expected to need.")


### Feature maps

The same microstructure passed through the trained network, block by block. Early maps still look
like the microstructure with the boundaries picked out. Later maps are coarse and abstract, and
there is no reason to expect any single channel to have a name.

In [ ]:
# --- Feature maps through the trained network --------------------------------
probe_i = int(np.argmax(np.abs(yd_te)))                 # the most anisotropic test case
probe_img = Xte_img[probe_i]
probe_t = torch.tensor(probe_img).view(1, 1, 64, 64)

acts, h = [], probe_t
for layer in model_dE:
    h = layer(h)
    if isinstance(layer, nn.MaxPool2d):
        acts.append(h.detach().numpy()[0])

fig, axes = plt.subplots(3, 7, figsize=(14, 6.4))
for r, maps in enumerate(acts):
    axes[r, 0].imshow(probe_img, cmap="gray", interpolation="nearest")
    axes[r, 0].set_ylabel(f"block {r+1}\n{maps.shape}", fontsize=8)
    order = np.argsort(-maps.std(axis=(1, 2)))[:6]      # the six most active channels
    for c, k in enumerate(order):
        axes[r, c+1].imshow(maps[k], cmap="viridis", interpolation="nearest")
        axes[r, c+1].set_title(f"ch {k}", fontsize=8)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle(f"feature maps for one test microstructure, "
             f"$\\Delta E$ = {yd_te[probe_i]:+.2f} GPa", y=1.0, fontsize=11)
plt.tight_layout(); plt.show()


### The same forward pass, animated

The figure above is a snapshot of three blocks. The animation below walks the same microstructure
through every stage of the trained network in order, and adds the two numbers that define a
convolutional architecture: how many pixels a side each feature map has, and how many channels there
are.

Left: every channel at the current stage, drawn at its true relative size, so a map at half the
resolution is drawn half as wide. Each channel is scaled to its own range, otherwise the quiet ones
would be invisible. Right: the resolution and the channel count as the stage index advances.

Watch the tiles get smaller and more numerous. The ratio of the two is the design decision the
architecture makes at every block.

In [ ]:
# --- Animation: one microstructure through the trained network ---------------
# Same trained model and same microstructure as the figure above.
hcur = torch.tensor(probe_img).view(1, 1, 64, 64)
SP_NAMES = ["conv 1 + ReLU", "pool 1", "conv 2 + ReLU", "pool 2", "conv 3 + ReLU", "pool 3"]
SP = [("input", probe_img[None].astype(np.float32))]
ks = 0
dense_act = None
for layer in model_dE:
    hcur = layer(hcur)
    if hcur.dim() == 4 and isinstance(layer, (nn.ReLU, nn.MaxPool2d)):
        SP.append((SP_NAMES[ks], hcur.detach().numpy()[0])); ks += 1
    if hcur.dim() == 2 and isinstance(layer, nn.ReLU):
        dense_act = hcur.detach().numpy()[0]
pred_probe = float(hcur.detach().numpy().ravel()[0])

SNAME = [s[0] for s in SP] + ["dense 64 + ReLU", "output"]
CH    = [1, 16, 16, 32, 32, 64, 64, 64, 1]
SPAT  = [64, 64, 32, 32, 16, 16, 8, 1, 1]
NVAL  = [c * s * s for c, s in zip(CH, SPAT)]
N_ST  = len(SNAME)                      # 9 stages
FPS_H = 6
N_FR_H = N_ST * FPS_H                   # 54 frames

print(f"{'stage':18s} {'channels':>9s} {'size':>6s} {'values':>9s}")
for nm, c, s, v in zip(SNAME, CH, SPAT, NVAL):
    print(f"{nm:18s} {c:>9d} {s:>6d} {v:>9,d}")
print(f"\nprediction for this microstructure  {pred_probe:+.3f} GPa")
print(f"true value                          {yd_te[probe_i]:+.3f} GPa")

GAP = 2
def tile_layout(n, S):
    cols = 1 if n == 1 else (4 if n <= 16 else 8)
    rows = int(np.ceil(n / cols))
    return rows, cols, cols * S + (cols - 1) * GAP, rows * S + (rows - 1) * GAP

cmap_v = plt.cm.viridis
cmap_g = plt.cm.gray

def mosaic(a, nshow, cm):
    # Returns an RGBA mosaic. Channels not yet revealed are drawn light grey, so a
    # partly filled stage never reads as an empty box.
    n, S, _ = a.shape
    rows, cols, W, H = tile_layout(n, S)
    rgba = np.ones((H, W, 4), np.float32)
    order = np.argsort(-a.std(axis=(1, 2))) if n > 1 else np.arange(n)
    for t in range(n):
        ch = order[t]; r, c = t // cols, t % cols
        if t < nshow:
            tile = a[ch].astype(np.float32)
            lo, hi = float(tile.min()), float(tile.max())
            tile = (tile - lo) / (hi - lo) if hi > lo else np.zeros_like(tile)
            px = cm(tile)
        else:
            px = np.ones((S, S, 4), np.float32); px[..., :3] = 0.88
        rgba[r*(S+GAP):r*(S+GAP)+S, c*(S+GAP):c*(S+GAP)+S] = px
    return rgba, W, H

fig = plt.figure(figsize=(13.4, 5.6))
gsh = fig.add_gridspec(1, 2, width_ratios=[1.35, 1.0], wspace=0.22)
axM, axC = fig.add_subplot(gsh[0, 0]), fig.add_subplot(gsh[0, 1])
axM.grid(False); axM.set_xticks([]); axM.set_yticks([])
for sp in ("top", "right", "bottom", "left"):
    axM.spines[sp].set_visible(False)
axM.set_xlim(-8, 288); axM.set_ylim(300, -40)

im_map = axM.imshow(np.ones((2, 2, 4), np.float32), extent=(0, 64, 64, 0),
                    interpolation="nearest", zorder=2)
im_vec = axM.imshow(np.full((1, 64), np.nan), cmap=plt.cm.viridis, extent=(0, 256, 138, 122),
                    interpolation="nearest", zorder=2)
im_vec.set_visible(False)
frame_r = Rectangle((0, 0), 64, 64, fill=False, ec=C_FIT, lw=1.6, zorder=4)
axM.add_patch(frame_r)
out_box = Rectangle((0, 102), 56, 56, fc="#f2e4c9", ec=C_FIT, lw=1.8, zorder=3, visible=False)
axM.add_patch(out_box)
out_txt = axM.text(28, 130, "", ha="center", va="center", fontsize=13, zorder=5)
cap_h   = axM.text(-8, -30, "", ha="left", va="center", fontsize=11.5)
sub_h   = axM.text(-8, 292, "", ha="left", va="top", fontsize=9, color="0.35")

axC.set_yscale("log", base=2)
axC.set_xlim(-0.4, N_ST - 0.6); axC.set_ylim(0.55, 200)
axC.set_xticks(range(N_ST)); axC.set_xticklabels(SNAME, rotation=38, ha="right", fontsize=8)
axC.set_ylabel("count")
ln_s, = axC.plot([], [], "-o", color=C_DATA, lw=2, ms=5, drawstyle="steps-post",
                 label="spatial size, pixels per side")
ln_c, = axC.plot([], [], "-s", color=C_FIT, lw=2, ms=5, drawstyle="steps-post",
                 label="channels")
mk_s, = axC.plot([], [], "o", ms=11, mfc="none", mec=C_DATA, mew=2)
mk_c, = axC.plot([], [], "s", ms=11, mfc="none", mec=C_FIT, mew=2)
axC.legend(fontsize=8.5, loc="lower right")
axC.set_title("resolution falls, channel count rises", fontsize=10)
info_h = axC.text(0.98, 0.97, "", transform=axC.transAxes, ha="right", va="top", fontsize=9,
                  family="monospace", bbox=dict(fc="w", ec="0.75", boxstyle="round,pad=0.35"))

def update_hier(f):
    s, t = f // FPS_H, (f % FPS_H) / (FPS_H - 1)
    im_map.set_visible(False); im_vec.set_visible(False); out_box.set_visible(False)
    out_txt.set_text(""); frame_r.set_visible(True)
    if s < 7:
        a = SP[s][1]
        n = a.shape[0]
        nshow = max(1, int(np.ceil(t * n))) if n > 1 else 1
        M, W, H = mosaic(a, nshow, cmap_g if s == 0 else cmap_v)
        y0 = 130.0 - H / 2.0
        im_map.set_data(M); im_map.set_extent((0, W, y0 + H, y0))
        im_map.set_visible(True)
        frame_r.set_xy((0, y0)); frame_r.set_width(W); frame_r.set_height(H)
        sub_h.set_text(f"{nshow} of {n} channel" + ("s" if n > 1 else "") +
                       f" drawn, each {SPAT[s]} x {SPAT[s]} pixels")
        cap_h.set_text(f"stage {s+1} of {N_ST}:  {SNAME[s]}   ({CH[s]} x {SPAT[s]} x {SPAT[s]})")
    elif s == 7:
        nshow = max(1, int(np.ceil(t * 64)))
        v = dense_act.astype(np.float32).copy(); v[nshow:] = np.nan
        im_vec.set_data(v.reshape(1, 64))
        im_vec.set_clim(0, max(float(dense_act.max()), 1e-6)); im_vec.set_visible(True)
        frame_r.set_xy((0, 122)); frame_r.set_width(256); frame_r.set_height(16)
        sub_h.set_text("the 4096 values are flattened and mixed by a dense layer.\n"
                       "no spatial structure is left, only 64 numbers")
        cap_h.set_text(f"stage {s+1} of {N_ST}:  {SNAME[s]}")
    else:
        out_box.set_visible(True); frame_r.set_visible(False)
        out_txt.set_text(f"$\\Delta E$\n{pred_probe:+.2f}")
        sub_h.set_text(f"one number. true $\\Delta E$ = {yd_te[probe_i]:+.2f} GPa")
        cap_h.set_text(f"stage {s+1} of {N_ST}:  {SNAME[s]}")
    xs_ = np.arange(s + 1)
    ln_s.set_data(xs_, SPAT[:s+1]); ln_c.set_data(xs_, CH[:s+1])
    mk_s.set_data([s], [SPAT[s]]);  mk_c.set_data([s], [CH[s]])
    info_h.set_text(f"channels {CH[s]:>7d}\nsize     {SPAT[s]:>7d}\nvalues   {NVAL[s]:>7,d}")
    return []

fig.subplots_adjust(left=0.02, right=0.97, bottom=0.30, top=0.90)
anim_hier = animation.FuncAnimation(fig, update_hier, frames=N_FR_H, interval=210, blit=False)
plt.close(fig)
HTML(anim_hier.to_jshtml())


**What the animation shows.** Three things, and the printed table has the numbers.

**Resolution halves at every pool, and the channel count rises at every convolution.** The tiles shrink and
multiply. The number of stored values is not monotonic: it jumps at each convolution and falls at
each pool, and the printed `values` column shows exactly where.

**Early maps still look like the microstructure, late maps do not.** After conv 1 the fibre
boundaries are visible in most channels. After pool 3 each map is 8 by 8, one value for every 8 by 8
patch of the original cell, and no channel resembles a picture of anything.

**The head throws the geometry away.** The dense layer takes 4096 numbers with a known spatial
arrangement and returns 64 with none. Everything geometric the network is going to use must have
been extracted before that point.

Anyone who has built a multigrid hierarchy will recognise the shape of this: a sequence of
increasingly coarse representations of the same field. The resemblance stops at the shape. The
restriction operator here is a learned convolution followed by a maximum, there is no prolongation
and no coarse-grid correction, and nothing is solved.

### How far can one unit see

A unit in the final feature map does not see the whole cell. It sees a square patch of the input,
its **receptive field**, and everything outside that patch is invisible to it.

The size follows from the architecture alone. Starting from a single unit and working backwards, a
layer with kernel size $k$ and stride $s$ turns a field of $r$ into

$$\boxed{\;r_{\text{in}} \;=\; (r_{\text{out}} - 1)\,s + k\;}$$

Applying this to the six layers of the trunk, from `pool 3` back to `conv 1`, gives the number the
next cell prints. The cell then measures the same quantity directly: it perturbs one input pixel at
a time and records whether the chosen output position changes. Computed and measured must agree.

This matters for the physics. Part 3 measured which descriptor carries the anisotropy, and it is a
directional two-point correlation at the lag printed there as `LAG_TOP`. If the receptive field were
smaller than that lag, no unit in the final map could represent the quantity at all.

In [ ]:
# --- Receptive field: computed from the architecture, then measured ----------
trunk = nn.Sequential(*list(model_dE)[:9])      # the three conv blocks, without the head
trunk.eval()

LAY_RF = [("conv 1", 3, 1), ("pool 1", 2, 2), ("conv 2", 3, 1),
          ("pool 2", 2, 2), ("conv 3", 3, 1), ("pool 3", 2, 2)]
r_rf = 1
RF_STEPS = [("one output unit", 1)]
for nm, k_, s_ in reversed(LAY_RF):
    r_rf = (r_rf - 1) * s_ + k_
    RF_STEPS.append((f"back through {nm}", r_rf))
RF_COMPUTED = r_rf

print("receptive field, computed backwards from one output unit")
for nm, v in RF_STEPS:
    print(f"  {nm.replace(chr(10), ' '):34s} {v:>3d} x {v:<3d}")

# measure it: perturb one input pixel at a time and see whether this output position moves
t0 = time.time()
I_RF, J_RF = 4, 4                               # a central position in the 8x8 final map
base_rf = trunk(torch.tensor(probe_img).view(1, 1, 64, 64)).detach().numpy()[0][:, I_RF, J_RF]
rf_mask = np.zeros((64, 64), dtype=bool)
scan = range(20, 52)                            # a 32x32 window that contains the field
for delta in (1.0, -1.0):
    batch, idxs = [], []
    for p in scan:
        for q in scan:
            im2 = probe_img.copy(); im2[p, q] += delta
            batch.append(im2); idxs.append((p, q))
    Xb = torch.tensor(np.stack(batch)).view(-1, 1, 64, 64)
    outs = []
    for k0 in range(0, len(Xb), 256):
        outs.append(trunk(Xb[k0:k0+256]).detach().numpy()[:, :, I_RF, J_RF])
    outs = np.concatenate(outs)
    for n_, (p, q) in enumerate(idxs):
        if np.abs(outs[n_] - base_rf).max() > 1e-6:
            rf_mask[p, q] = True

rows_rf, cols_rf = np.where(rf_mask)
RF_ROWS = int(rows_rf.max() - rows_rf.min() + 1)
RF_COLS = int(cols_rf.max() - cols_rf.min() + 1)
CY_RF = 0.5 * (rows_rf.min() + rows_rf.max())
CX_RF = 0.5 * (cols_rf.min() + cols_rf.max())
LAG_RF = LAG_TOP                                # lag of the dominant descriptor, in pixels

print(f"\ncomputed from the architecture   {RF_COMPUTED} x {RF_COMPUTED} pixels")
print(f"measured bounding box             {RF_ROWS} x {RF_COLS} pixels")
print(f"pixels that change this unit     {int(rf_mask.sum())}   "
      f"(a full {RF_COMPUTED} x {RF_COMPUTED} square is {RF_COMPUTED**2})")
print(f"computed and measured agree:     {RF_ROWS == RF_COMPUTED and RF_COLS == RF_COMPUTED}")
print(f"measurement took {time.time()-t0:.1f} s")
print(f"\nreceptive field {RF_COMPUTED} px against descriptor lag {LAG_RF} px: "
      f"ratio {RF_COMPUTED/LAG_RF:.2f}")

r_nopool = 1
for nm, k_, s_ in reversed([l for l in LAY_RF if l[0].startswith("conv")]):
    r_nopool = (r_nopool - 1) * 1 + k_
print(f"the same three convolutions with no pooling would reach only {r_nopool} px")


### The same thing, animated

The animation starts from one unit of the final feature map and walks backwards through the six
layers, drawing on the input the square of pixels that can still reach it.

Left: the microstructure, with the field drawn as an orange square and the descriptor lag drawn as
a green bar for scale. At the last step the measured set of pixels is shaded, so the
computed square and the measured set can be compared directly. Right: the field size after each
step, with the two reference lines.

In [ ]:
# --- Animation: the receptive field growing back through the layers ----------
RF_SIZES = [v for _, v in RF_STEPS]
RF_LABS  = [nm for nm, _ in RF_STEPS]
FPS_R  = 7
N_FR_R = FPS_R * len(RF_SIZES)                  # 49 frames

fig = plt.figure(figsize=(12.6, 5.2))
gsr = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.05], wspace=0.26)
axI, axG = fig.add_subplot(gsr[0, 0]), fig.add_subplot(gsr[0, 1])
axI.grid(False); axI.set_xticks([]); axI.set_yticks([])
axI.imshow(probe_img, cmap="gray", interpolation="nearest")
mask_rgba = np.ma.masked_where(~rf_mask, np.ones_like(probe_img))
im_rf = axI.imshow(mask_rgba, cmap="autumn", alpha=0.0, interpolation="nearest", vmin=0, vmax=1)
box_rf = Rectangle((0, 0), 1, 1, fill=False, ec=C_FIT, lw=2.6, zorder=5)
axI.add_patch(box_rf)
axI.plot([CX_RF], [CY_RF], "x", color=C_FIT, ms=9, mew=2.2, zorder=6)
axI.plot([2, 2 + LAG_RF], [4.5, 4.5], "-", color=C_ALT, lw=4.0,
         solid_capstyle="butt", zorder=6)
axI.text(2, 8.0, f"{LAG_RF} px, descriptor lag", color=C_ALT, fontsize=8.5, va="top", zorder=6,
         bbox=dict(fc="w", ec="none", alpha=0.85, boxstyle="round,pad=0.15"))
axI.set_title("the microstructure, 64 x 64", fontsize=10)
cap_rf = axI.text(0.0, 68.5, "", fontsize=10.5, ha="left", va="top")

axG.set_xlim(-0.5, len(RF_SIZES) - 0.5); axG.set_ylim(0, 26)
axG.set_xticks(range(len(RF_SIZES)))
axG.set_xticklabels(RF_LABS, rotation=38, ha="right", fontsize=8)
axG.set_ylabel("receptive field, pixels per side")
ln_rf,  = axG.plot([], [], "-o", color=C_FIT, lw=2.2, ms=6)
mk_rf,  = axG.plot([], [], "o", ms=12, mfc="none", mec=C_FIT, mew=2)
axG.axhline(LAG_RF, color=C_ALT, ls="--", lw=1.4)
axG.text(-0.35, LAG_RF + 0.5, f"lag {LAG_RF}, the length scale of\nthe dominant descriptor",
         ha="left", va="bottom", fontsize=8, color=C_ALT)
axG.axhline(RF_COMPUTED, color="0.4", ls=":", lw=1.3)
axG.text(-0.35, RF_COMPUTED + 0.5, f"{RF_COMPUTED} px", ha="left", fontsize=8.5, color="0.35")
axG.set_title("the field grows going back through the layers", fontsize=10)
info_rf = axG.text(0.98, 0.04, "", transform=axG.transAxes, ha="right", va="bottom",
                   fontsize=9, family="monospace",
                   bbox=dict(fc="w", ec="0.75", boxstyle="round,pad=0.3"))

def update_rf(f):
    s, t = f // FPS_R, (f % FPS_R) / (FPS_R - 1)
    prev = RF_SIZES[s - 1] if s > 0 else RF_SIZES[0]
    cur  = RF_SIZES[s]
    size = prev + (cur - prev) * t
    box_rf.set_xy((CX_RF - size / 2, CY_RF - size / 2))
    box_rf.set_width(size); box_rf.set_height(size)
    ln_rf.set_data(np.arange(s + 1), RF_SIZES[:s+1]); mk_rf.set_data([s], [cur])
    cap_rf.set_text(f"{RF_LABS[s].replace(chr(10), ' ')}:  {cur} x {cur} pixels")
    if s == len(RF_SIZES) - 1 and t > 0.5:
        im_rf.set_alpha(0.35)
        info_rf.set_text(f"computed  {RF_COMPUTED} x {RF_COMPUTED}\n"
                         f"measured  {RF_ROWS} x {RF_COLS}\n"
                         f"pixels    {int(rf_mask.sum())}")
    else:
        im_rf.set_alpha(0.0); info_rf.set_text("")
    return []

fig.subplots_adjust(left=0.03, right=0.98, bottom=0.30, top=0.90)
anim_rf = animation.FuncAnimation(fig, update_rf, frames=N_FR_R, interval=180, blit=False)
plt.close(fig)
HTML(anim_rf.to_jshtml())


**What the animation shows.** The field doubles at every pooling layer and grows by two
pixels at every 3x3 convolution. The pools do almost all the work, and the cell printed what the
same three convolutions would reach without them.

The shaded set at the final step is the measured one, and it reaches the edges of the computed
square in both directions, which is why the measured and computed sizes agree. Inside the square the
shading has small gaps: the printed count of pixels that move the unit is a little short of the full
square, because a ReLU that is off or a max pool that selects a different element blocks that
particular path. The formula gives the largest field the architecture allows, and the measurement
gives what is live at this input.

The practical reading is the comparison with the green bar. The descriptor that carries the
anisotropy lives at the lag printed above, and one unit of the final feature map sees further than
that, so the architecture is at least capable of representing the relevant length scale. Being
capable of it is not the same as learning it, and the scores earlier in this notebook show the
network does not fully recover what the descriptor model gets.

### Saliency

Saliency is the gradient of the predicted output with respect to the input pixels. A pixel with a
large absolute gradient is one where a small change would move the prediction most. It is computed
with one backward pass, which is why it is the cheapest interpretability tool there is.

Read it carefully, and read the caveats below the figure before drawing any conclusion from it.

In [ ]:
# --- Gradient saliency --------------------------------------------------------
def saliency(model, img, smooth=1.5):
    x = torch.tensor(img).view(1, 1, 64, 64).requires_grad_(True)
    model.zero_grad()
    out = model(x)
    out.backward()
    g = x.grad.detach().numpy()[0, 0]
    return ndimage.gaussian_filter(np.abs(g), smooth)

order_te = np.argsort(np.abs(yd_te))[::-1][:3]          # three strongly anisotropic cases
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for c, i in enumerate(order_te):
    img = Xte_img[i]
    s = saliency(model_dE, img)
    axes[0, c].imshow(img, cmap="gray", interpolation="nearest")
    axes[0, c].set_title(f"true $\\Delta E$ = {yd_te[i]:+.2f}\n"
                         f"predicted {res_dE['pred'][i]:+.2f} GPa", fontsize=9)
    axes[1, c].imshow(img, cmap="gray", interpolation="nearest", alpha=0.85)
    axes[1, c].imshow(s, cmap="inferno", alpha=0.55, interpolation="bilinear")
    axes[1, c].set_title("saliency, smoothed", fontsize=9)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()

# how much of the sensitivity sits on fibre boundaries rather than in the bulk
img0 = Xte_img[order_te[0]]
s0   = saliency(model_dE, img0)
edge = ndimage.binary_dilation(
    ndimage.laplace(img0) != 0, iterations=1)
print(f"fraction of pixels on or next to a fibre boundary: {edge.mean():.3f}")
print(f"fraction of total saliency there:                  {s0[edge].sum()/s0.sum():.3f}")


### How much to believe

The printed numbers above compare the share of pixels lying on or beside a fibre boundary with the
share of the total saliency that falls there. Sensitivity is somewhat concentrated on the
boundaries, which is consistent with a model that responds to the arrangement of interfaces. It is a
weak, reassuring sign and nothing more.

Three warnings, in order of importance.

1. **Saliency measures sensitivity, not cause.** It says the output would change if that pixel
   changed. It does not say the network used that region to reach its answer, and it says nothing
   about whether the underlying physics depends on it.
2. **It is a local, first-order quantity.** The gradient is taken at one image. A different
   microstructure gives a different map, and a binary image cannot actually be changed by a small
   amount in the way the derivative assumes.
3. **A model that predicts badly still produces a confident looking saliency map.** The network in
   this notebook leaves roughly a quarter of the variance in $\Delta E$ unexplained, and scores
   below the descriptor model. Any interpretation of its internals inherits that limitation.

Used as a sanity check, saliency is worth the one backward pass it costs. Used as evidence about the
mechanics, it is not evidence.

---

# What to take away

1. A convolution is a small kernel slid over the image, and everything that matters about it follows
   from that: locality, weight sharing, and a parameter count set by the size of the pattern rather
   than the size of the image.
2. Padding and stride are not cosmetic. Circular padding is the correct choice for a periodic cell,
   and the output shape formula is worth being able to write down without a computer.
3. The permutation test bites only when the target needs geometry. On $E_{mean}$ neither model
   moved, because that target is close to a pixel count and shuffling preserves the count. On
   $\Delta E$ the CNN fell from its trained score to about zero while the dense network had nothing
   to lose. Choose the target before concluding anything from the test.
4. On $E_{mean}$ everything works, because the target is nearly a function of the volume fraction.
   Choose a test problem where the methods can actually differ before concluding anything.
5. On $\Delta E$ the CNN lost to fourteen hand-built descriptors fitted in a fraction of the time,
   and lost by more to the full bank of thirty. All three read the same 64x64 images, so the
   comparison is like for like. Deep learning is not automatically the stronger option, and a
   feature set designed by someone who understands the mechanics is a serious baseline.
6. The gap to the published tuned CNN on this target was closed by a directional architecture and a
   much larger training budget, not by switching to a CNN as such. Architecture and budget are part
   of the method, and both belong in the reported result.

---

# Exercises

### 1. Predict the output shape, then check it
For a 96x96 input, a 5x5 kernel, padding 2 and stride 2, write down the output size from the formula
in Part 1e. Then build the layer with `nn.Conv2d` and confirm it. Repeat for three blocks of
convolution and 2x2 pooling, and say what the final spatial size is.

### 2. Design a kernel by hand
Using the kernel editor in Part 1d, build a 3x3 kernel that responds strongly to fibre boundaries
running at 45 degrees and weakly to horizontal and vertical ones. Explain, from the sign pattern, why
it does what it does. What does its sum have to be?

### 3. The descriptors were better
Report the test $R^2$, the fitting time and the number of parameters for the 14 descriptor model, the
30 descriptor model and the CNN on $\Delta E$. State plainly which you would put into a design loop, and what would have to
change for the other answer to be right. Then say what the descriptor route costs that the table does
not show.

### 4. Break the CNN on purpose
Retrain the CNN on $\Delta E$ with `augmented=False`, and again with zero padding instead of
circular. Report both scores against the run in Part 3. Which of the two changes costs more, and does
that match what you expected from the periodicity of the cells?

### 5. Rotation is not a free symmetry
Part 2 left out 90 degree rotation because it exchanges $E_{22}$ and $E_{33}$. Add it to `augment`,
flipping the sign of the $\Delta E$ target whenever it is applied, and retrain. Does the extra
augmentation help? Now add it without flipping the sign and report what happens, so you have seen the
failure mode.

### 6. Close some of the gap
The head of this network flattens a 64x8x8 tensor into a dense layer, which is where most of the
parameters are and which ties the answer to absolute position. Replace it with a global average pool
over the spatial dimensions followed by a small dense layer, retrain on $\Delta E$, and report the
score and the parameter count. Then argue, from what the target is, why that change might help.
